# 02 - Dataset Review and Split

This notebook continues the project using the **CRISP-DM** workflow.

At this stage, we are still in **Data Understanding** and **Data Preparation**. The goal is to review the labelled dataset created in `01_data_prep.ipynb`, confirm that the labels and image paths look usable, and prepare a clean table that can later be split for model training.

We will keep the notebook simple and readable. Each section reloads what it needs from saved files, so the notebook can be closed, reopened, and continued without rerunning everything before it.

## How we will work in this notebook

We will not write the whole machine learning pipeline at once.

Instead, we will move in small, checkable sections:

1. Load the saved data-prep outputs.
2. Confirm the expected columns exist.
3. Check how many labelled images are usable.
4. Build a clean model-candidate dataset.
5. Save that table for the next notebook section.

After running this notebook section, we will inspect the results before creating the final train, validation, and test split.

## Install required libraries

Run this once if the notebook environment does not already have the required packages.

The project mainly needs `pandas` for tables, `numpy` for small numeric checks, `matplotlib` for charts, `scikit-learn` for splitting later, and `Pillow` for image checks.

In [ ]:
%pip install pandas numpy matplotlib scikit-learn pillow

## Section 1: Set up paths and imports

This cell keeps all project paths in one place.

The notebook expects the processed CSV files from `01_data_prep.ipynb` to already exist in `data/processed/`.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR

## Section 2: Load the prepared tables

The first notebook saved two important tables:

- `irr_image_labels.csv`: one row per image/IRR record, with the main label attached.
- `irr_pattern_entries.csv`: one row per pattern entry found in field `9.307`.

For model training, the image-level table is the main table. The pattern-entry table remains useful for checking multi-label or repeated-label cases.

In [ ]:
image_labels_path = PROCESSED_DIR / "irr_image_labels.csv"
pattern_entries_path = PROCESSED_DIR / "irr_pattern_entries.csv"
label_counts_path = PROCESSED_DIR / "pattern_label_counts.csv"

for path in [image_labels_path, pattern_entries_path, label_counts_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected file: {path}")

In [ ]:
id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

image_labels = pd.read_csv(image_labels_path, dtype=id_columns)
pattern_entries = pd.read_csv(pattern_entries_path, dtype=id_columns)
label_counts = pd.read_csv(label_counts_path)

print("Image-level rows:", len(image_labels))
print("Pattern-entry rows:", len(pattern_entries))
print("Label-count rows:", len(label_counts))

## Section 3: Check the columns we need

Before doing any analysis, we confirm that the table has the fields needed for classification.

The important fields are:

- `primary_label`: the raw SD302 pattern label chosen for the image.
- `broad_class`: the broader class, such as arch, loop, or whorl.
- `subtype`: the subtype where the raw label gives one.
- `png_path`: the linked fingerprint image.
- `png_found`: whether the image file was found in the workspace.

In [ ]:
required_columns = [
    "subject_id",
    "finger_position",
    "primary_label",
    "broad_class",
    "subtype",
    "png_path",
    "png_found",
]

missing_columns = [column for column in required_columns if column not in image_labels.columns]
missing_columns

In [ ]:
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")

## Section 4: Review the available labels

This shows the labels extracted from the raw SD302g records.

The counts help us decide what kind of classifier is realistic. Broad classes should be more stable. Some subtypes have very small counts, so they must be handled carefully.

In [ ]:
label_counts

In [ ]:
broad_class_counts = image_labels["broad_class"].value_counts().sort_index()
broad_class_counts

In [ ]:
ax = broad_class_counts.plot(kind="bar", figsize=(7, 4))
ax.set_title("Available Images by Broad Pattern Class")
ax.set_xlabel("Broad class")
ax.set_ylabel("Number of images")
plt.xticks(rotation=0)
plt.show()

## Section 5: Keep only rows with linked images

A row is useful for image classification only if the image file exists.

Here we create a clean candidate table for modelling. This is not the final training split yet; it is the cleaned table we will inspect first.

In [ ]:
model_candidates = image_labels.copy()

model_candidates = model_candidates[model_candidates["png_found"] == True]
model_candidates = model_candidates.dropna(subset=["png_path", "broad_class"])

print("Usable image rows:", len(model_candidates))

In [ ]:
model_candidates[[
    "subject_id",
    "finger_position",
    "primary_label",
    "broad_class",
    "subtype",
    "png_path",
]].head()

## Section 6: Save the clean model-candidate table

This table becomes the handoff point for the next step.

Saving it means the next notebook section can start from this file directly instead of depending on variables in memory.

In [ ]:
model_candidates_path = PROCESSED_DIR / "model_candidates.csv"

model_candidates.to_csv(model_candidates_path, index=False)

print("Saved:", model_candidates_path)
print("Rows:", len(model_candidates))

## Stop and review

Pause here after running the cells above.

The important things to check are:

- Are all required columns present?
- How many usable image rows were found?
- Are the broad classes balanced enough for a first classifier?
- Does the sample table show realistic image paths and labels?

Once these results look correct, the next section will create a subject-aware train, validation, and test split.

## Section 7: Create the first modelling dataset

For the first classifier, we will use the classes with enough examples to train and evaluate properly:

- `arch`
- `left_slant_loop`
- `right_slant_loop`
- `whorl`

The `unclassifiable` group has only a small number of images, so we will keep it out of the first supervised model. We can return to it later as a reject/unknown class or as a separate analysis point.

In [ ]:
from pathlib import Path

import pandas as pd

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

model_candidates_path = PROCESSED_DIR / "model_candidates.csv"

In [ ]:
id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

model_candidates = pd.read_csv(model_candidates_path, dtype=id_columns)

print("Loaded rows:", len(model_candidates))

In [ ]:
broad_classes_for_first_model = [
    "arch",
    "left_slant_loop",
    "right_slant_loop",
    "whorl",
]

broad_model_data = model_candidates[
    model_candidates["broad_class"].isin(broad_classes_for_first_model)
].copy()

broad_model_data["broad_class"].value_counts().sort_index()

## Section 8: Split by subject, not by image

This is important for academic quality.

If images from the same subject appear in both training and testing, the model may partly learn the person instead of learning the fingerprint pattern. To reduce that risk, we split using `subject_id` as a group.

The split will be:

- 70% training
- 15% validation
- 15% testing

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42,
)

train_index, temp_index = next(
    first_split.split(broad_model_data, groups=broad_model_data["subject_id"])
)

In [ ]:
train_data = broad_model_data.iloc[train_index].copy()
temp_data = broad_model_data.iloc[temp_index].copy()

second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42,
)

validation_index, test_index = next(
    second_split.split(temp_data, groups=temp_data["subject_id"])
)

In [ ]:
validation_data = temp_data.iloc[validation_index].copy()
test_data = temp_data.iloc[test_index].copy()

train_data["split"] = "train"
validation_data["split"] = "validation"
test_data["split"] = "test"

split_data = pd.concat([train_data, validation_data, test_data], ignore_index=True)

split_data["split"].value_counts()

## Section 9: Check the split quality

After splitting, we check two things:

1. No subject appears in more than one split.
2. Each split still contains examples from all target classes.

In [ ]:
subject_split_counts = split_data.groupby("subject_id")["split"].nunique()
leaked_subjects = subject_split_counts[subject_split_counts > 1]

print("Subjects appearing in more than one split:", len(leaked_subjects))

In [ ]:
class_counts_by_split = pd.crosstab(
    split_data["broad_class"],
    split_data["split"],
)

class_counts_by_split

## Section 10: Save the split dataset

This saved file is the starting point for model training.

The next notebook can load this file directly and train the first broad fingerprint pattern classifier.

In [ ]:
split_data_path = PROCESSED_DIR / "broad_model_split.csv"

split_data.to_csv(split_data_path, index=False)

print("Saved:", split_data_path)
print("Rows:", len(split_data))

## Stop and review before modelling

Pause here after running the split section.

Before training a model, we should confirm:

- `Subjects appearing in more than one split` is `0`.
- all four target classes appear in train, validation, and test.
- the class counts are acceptable for a first academic baseline model.

## Section 11: Load the saved split for analysis

This section can run on its own after `broad_model_split.csv` has been created.

The aim is to analyse the split in a more academic way before modelling. We check dataset size, class balance, subject leakage, source distribution, and label complexity.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROCESSED_DIR / "figures"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
split_data_path = PROCESSED_DIR / "broad_model_split.csv"

id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

split_data = pd.read_csv(split_data_path, dtype=id_columns)

print("Loaded split rows:", len(split_data))

## Section 12: Overall split summary

This table shows how many images and subjects are in each split.

For an academic machine learning project, this matters because the split must be large enough to train, tune, and test the model fairly.

In [ ]:
split_summary = (
    split_data.groupby("split")
    .agg(images=("png_path", "count"), subjects=("subject_id", "nunique"))
    .reindex(["train", "validation", "test"])
)

split_summary["image_percent"] = (split_summary["images"] / len(split_data) * 100).round(1)
split_summary

In [ ]:
ax = split_summary["images"].plot(kind="bar", figsize=(7, 4), color="#4C78A8")
ax.set_title("Number of Images in Each Dataset Split")
ax.set_xlabel("Split")
ax.set_ylabel("Number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "split_image_counts.png", dpi=150)
plt.show()

## Section 13: Class distribution by split

This is the most important split check.

The model should see every target class during training, validation, and testing. If a class is missing from validation or test, the evaluation would be weak.

In [ ]:
class_counts_by_split = pd.crosstab(
    split_data["broad_class"],
    split_data["split"],
)

class_counts_by_split = class_counts_by_split[["train", "validation", "test"]]
class_counts_by_split

In [ ]:
ax = class_counts_by_split.plot(kind="bar", figsize=(9, 5))
ax.set_title("Class Counts by Dataset Split")
ax.set_xlabel("Fingerprint pattern class")
ax.set_ylabel("Number of images")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_counts_by_split.png", dpi=150)
plt.show()

## Section 14: Class percentages inside each split

Raw counts are useful, but percentages make the split easier to compare.

If the percentages are very different between train, validation, and test, the model evaluation may become less stable.

In [ ]:
class_percent_by_split = class_counts_by_split.div(
    class_counts_by_split.sum(axis=0),
    axis=1,
) * 100

class_percent_by_split.round(1)

In [ ]:
ax = class_percent_by_split.T.plot(kind="bar", stacked=True, figsize=(8, 5))
ax.set_title("Class Percentage Composition by Split")
ax.set_xlabel("Split")
ax.set_ylabel("Percentage of images")
ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_percent_by_split.png", dpi=150)
plt.show()

## Section 15: Class imbalance check

Class imbalance tells us whether some classes are much easier for the model to learn simply because they appear more often.

The imbalance ratio is calculated as:

`largest class count / smallest class count`

A higher value means stronger imbalance.

In [ ]:
overall_class_counts = split_data["broad_class"].value_counts().sort_values(ascending=False)

imbalance_ratio = overall_class_counts.max() / overall_class_counts.min()

print("Overall class counts")
display(overall_class_counts)
print("Imbalance ratio:", round(imbalance_ratio, 2))

## Section 16: Subject-level split validation

A subject-aware split is important because the same subject may contribute multiple finger images.

If the same subject appears in both training and testing, the model may get an unfair advantage. This check confirms whether subject leakage exists.

In [ ]:
subject_split_counts = split_data.groupby("subject_id")["split"].nunique()
leaked_subjects = subject_split_counts[subject_split_counts > 1]

print("Subjects appearing in more than one split:", len(leaked_subjects))

In [ ]:
images_per_subject = split_data.groupby(["split", "subject_id"]).size().reset_index(name="images")

images_per_subject.groupby("split")["images"].describe().round(2)

In [ ]:
ax = images_per_subject.boxplot(column="images", by="split", figsize=(7, 4))
ax.set_title("Images per Subject by Split")
ax.set_xlabel("Split")
ax.set_ylabel("Images per subject")
plt.suptitle("")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "images_per_subject_by_split.png", dpi=150)
plt.show()

## Section 17: Source and capture-type checks

This project uses images linked from the available SD302 workspace.

Before modelling, we check whether the split is dominated by one collection source or capture type. This helps us describe the dataset honestly in the methodology.

In [ ]:
source_counts = pd.crosstab(split_data["collection_type"], split_data["split"])
source_counts = source_counts[["train", "validation", "test"]]
source_counts

In [ ]:
capture_counts = pd.crosstab(split_data["capture_type"], split_data["split"])
capture_counts = capture_counts[["train", "validation", "test"]]
capture_counts

In [ ]:
ax = capture_counts.T.plot(kind="bar", stacked=True, figsize=(8, 5))
ax.set_title("Capture Type Composition by Split")
ax.set_xlabel("Split")
ax.set_ylabel("Number of images")
ax.legend(title="Capture type")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "capture_type_by_split.png", dpi=150)
plt.show()

## Section 18: Label complexity check

Some records have more than one `9.307` pattern entry.

For the first model, we use `primary_label`, but we still report multi-label records because they may represent harder or more ambiguous examples. This is useful for the limitation section of the project.

In [ ]:
label_complexity_counts = pd.crosstab(
    split_data["num_pattern_labels"],
    split_data["split"],
)

label_complexity_counts = label_complexity_counts[["train", "validation", "test"]]
label_complexity_counts

In [ ]:
multi_label_rate = (split_data["num_pattern_labels"] > 1).mean() * 100

print("Records with multiple pattern labels:", round(multi_label_rate, 1), "%")

## Section 19: Academic interpretation of the split

Use the printed summary below as a draft interpretation for the report.

The wording is intentionally plain so it can be adapted into the methodology or results chapter.

In [ ]:
total_images = len(split_data)
total_subjects = split_data["subject_id"].nunique()
smallest_class = overall_class_counts.idxmin()
largest_class = overall_class_counts.idxmax()

print(f"The broad-pattern modelling dataset contains {total_images} images from {total_subjects} subjects.")
print(f"The largest class is {largest_class} with {overall_class_counts.max()} images.")
print(f"The smallest class is {smallest_class} with {overall_class_counts.min()} images.")
print(f"The overall class imbalance ratio is {imbalance_ratio:.2f}:1.")
print(f"Subject leakage across train, validation, and test splits: {len(leaked_subjects)} subjects.")
print(f"Multi-label records represent {multi_label_rate:.1f}% of the modelling dataset.")

## Stop and discuss the analysis

Pause here and review the tables and plots.

For the report, the key points to discuss are:

- the split is subject-aware, which protects the evaluation from subject leakage.
- the four target classes are present in all splits.
- the `arch` class is smaller than the loop and whorl classes, so class imbalance must be handled during training.
- some records contain multiple pattern labels, so the first classifier should be presented as a broad-pattern baseline using the selected `primary_label`.
- the next modelling notebook should report accuracy, precision, recall, F1-score, and confusion matrix rather than accuracy alone.

## Section 20: Create a roll-only modelling dataset

The earlier analysis showed that the dataset is dominated by rolled fingerprint images.

For the first serious model, we will therefore use only `roll` images. This gives the project a clearer academic scope:

> broad dermatoglyphic pattern classification from rolled fingerprint images.

This section can run on its own after `model_candidates.csv` has been created.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

current_folder = Path.cwd()
PROJECT_ROOT = current_folder.parent if current_folder.name == "notebooks" else current_folder
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROCESSED_DIR / "figures"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
id_columns = {
    "subject_id": "string",
    "finger_position": "string",
    "resolution": "string",
}

model_candidates = pd.read_csv(
    PROCESSED_DIR / "model_candidates.csv",
    dtype=id_columns,
)

print("Loaded candidate rows:", len(model_candidates))

In [ ]:
target_classes = ["arch", "left_slant_loop", "right_slant_loop", "whorl"]

roll_model_data = model_candidates[
    (model_candidates["capture_type"] == "roll")
    & (model_candidates["broad_class"].isin(target_classes))
].copy()

roll_model_data["broad_class"].value_counts().sort_index()

## Section 21: Create a subject-aware roll-only split

We still split by `subject_id`, not by image.

This protects the evaluation from subject leakage. The same person should not appear in both training and testing.

To improve class balance, we try several random subject-aware splits and keep the best one.

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def make_subject_split(data, random_state):
    first_split = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=random_state)
    train_index, temp_index = next(first_split.split(data, groups=data["subject_id"]))

    train_data = data.iloc[train_index].copy()
    temp_data = data.iloc[temp_index].copy()

    second_split = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=random_state)
    validation_index, test_index = next(second_split.split(temp_data, groups=temp_data["subject_id"]))

    validation_data = temp_data.iloc[validation_index].copy()
    test_data = temp_data.iloc[test_index].copy()

    train_data["split"] = "train"
    validation_data["split"] = "validation"
    test_data["split"] = "test"

    return pd.concat([train_data, validation_data, test_data], ignore_index=True)

In [ ]:
def class_balance_score(split_data):
    counts = pd.crosstab(split_data["broad_class"], split_data["split"])
    counts = counts.reindex(index=target_classes, columns=["train", "validation", "test"], fill_value=0)

    split_percent = counts.div(counts.sum(axis=0), axis=1)
    overall_percent = counts.sum(axis=1) / counts.values.sum()

    return split_percent.sub(overall_percent, axis=0).abs().sum().sum()

In [ ]:
best_score = float("inf")
best_split_data = None
best_random_state = None

for random_state in range(200):
    candidate_split = make_subject_split(roll_model_data, random_state)
    score = class_balance_score(candidate_split)

    if score < best_score:
        best_score = score
        best_split_data = candidate_split
        best_random_state = random_state

roll_split_data = best_split_data

print("Best random state:", best_random_state)
print("Balance score:", round(best_score, 4))

## Section 22: Check the roll-only split

Before saving the split, we check whether the same subject appears in more than one split and whether all four target classes are present.

In [ ]:
roll_subject_split_counts = roll_split_data.groupby("subject_id")["split"].nunique()
roll_leaked_subjects = roll_subject_split_counts[roll_subject_split_counts > 1]

print("Subjects appearing in more than one split:", len(roll_leaked_subjects))

In [ ]:
roll_class_counts_by_split = pd.crosstab(
    roll_split_data["broad_class"],
    roll_split_data["split"],
)

roll_class_counts_by_split = roll_class_counts_by_split[["train", "validation", "test"]]
roll_class_counts_by_split

In [ ]:
roll_split_summary = (
    roll_split_data.groupby("split")
    .agg(images=("png_path", "count"), subjects=("subject_id", "nunique"))
    .reindex(["train", "validation", "test"])
)

roll_split_summary["image_percent"] = (roll_split_summary["images"] / len(roll_split_data) * 100).round(1)
roll_split_summary

## Section 23: Save the roll-only split

This file will be the clean input for the first model training notebook.

In [ ]:
roll_split_path = PROCESSED_DIR / "roll_broad_model_split.csv"

roll_split_data.to_csv(roll_split_path, index=False)

print("Saved:", roll_split_path)
print("Rows:", len(roll_split_data))

## Section 24: Plot the roll-only split

These plots are for the methodology and data preparation discussion. They show how the final roll-only dataset is arranged before modelling.

In [ ]:
ax = roll_split_summary["images"].plot(kind="bar", figsize=(7, 4), color="#4C78A8")
ax.set_title("Roll-Only Dataset: Images by Split")
ax.set_xlabel("Split")
ax.set_ylabel("Number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "roll_split_image_counts.png", dpi=150)
plt.show()

In [ ]:
ax = roll_class_counts_by_split.plot(kind="bar", figsize=(9, 5))
ax.set_title("Roll-Only Dataset: Class Counts by Split")
ax.set_xlabel("Fingerprint pattern class")
ax.set_ylabel("Number of images")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "roll_class_counts_by_split.png", dpi=150)
plt.show()

In [ ]:
roll_class_percent_by_split = roll_class_counts_by_split.div(
    roll_class_counts_by_split.sum(axis=0),
    axis=1,
) * 100

roll_class_percent_by_split.round(1)

In [ ]:
ax = roll_class_percent_by_split.T.plot(kind="bar", stacked=True, figsize=(8, 5))
ax.set_title("Roll-Only Dataset: Class Percentage Composition")
ax.set_xlabel("Split")
ax.set_ylabel("Percentage of images")
ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "roll_class_percent_by_split.png", dpi=150)
plt.show()

## Section 25: Roll-only split interpretation

This summary explains what the roll-only split means for the project.

In [ ]:
roll_class_counts = roll_split_data["broad_class"].value_counts().sort_values(ascending=False)
roll_imbalance_ratio = roll_class_counts.max() / roll_class_counts.min()

print(f"The roll-only modelling dataset contains {len(roll_split_data)} images.")
print(f"It contains {roll_split_data['subject_id'].nunique()} unique subjects.")
print(f"The largest class is {roll_class_counts.idxmax()} with {roll_class_counts.max()} images.")
print(f"The smallest class is {roll_class_counts.idxmin()} with {roll_class_counts.min()} images.")
print(f"The class imbalance ratio is {roll_imbalance_ratio:.2f}:1.")
print(f"Subject leakage across splits: {len(roll_leaked_subjects)} subjects.")

## Stop before model training

Pause here and review the roll-only split.

If the split looks acceptable, the next notebook will be:

`03_model_training_baseline.ipynb`

That notebook will train the first broad classifier for rolled fingerprint images.